In [ ]:
import os, json
from copy import deepcopy

import numpy as np
try:
    from google import genai
except ImportError as exc:
    raise ImportError("Install the Gemini SDK first: pip install google-genai") from exc
from tqdm import tqdm

from mistakes_const import PARAPHRASE_PROMPT, ADD_MISTAKE_FEWSHOT
from repro import config as cfg
from util import store_jsonl, load_results

runs = cfg.RUNS

In [ ]:
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running this notebook.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
def query_api(prompt, client, model=GEMINI_MODEL):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
    )
    text = (response.text or "").strip()
    if not text:
        raise RuntimeError(f"Gemini returned no text for model {model}.")
    return {
        "text": text,
        "model": getattr(response, "model_version", model),
    }

In [ ]:
def make_question(question, options):
    _options = '\n'.join(["(" + o for o in options])
    
    return f"{question}\n\n{_options}"

In [ ]:
SOURCE_ROOT = 'mid_results'
PATH_ROOT = 'mistake_results'

for model_name, dataset, lr in runs:
    source_file = (
        f"{SOURCE_ROOT}/{dataset}/{model_name}/"
        f"{cfg.METHOD}_{cfg.STRATEGY}_s={cfg.STEPWISE}_lr={lr}_rs={cfg.SEED}"
        f"_pos={cfg.POS_FILTER}_ff2={cfg.FF2_ONLY}.out"
    )
    if not os.path.exists(source_file):
        print(f"Missing source file, skipping: {source_file}")
        continue

    resdir = f"{PATH_ROOT}/{dataset}/{model_name}"
    os.makedirs(resdir, exist_ok=True)
    path_to_store = (
        f"{resdir}/{cfg.METHOD}_{cfg.STRATEGY}_s={cfg.STEPWISE}_lr={lr}_rs={cfg.SEED}"
        f"_pos={cfg.POS_FILTER}_ff2={cfg.FF2_ONLY}_mistakes.jsonl"
    )

    if os.path.exists(path_to_store):
        print(f"Results exist, skipping: {path_to_store}")
        continue

    print(f"Running for {dataset} & {model_name}")
    results = load_results(source_file)
    augmented_results = deepcopy(results)

    for idx, instance in tqdm(enumerate(results), total=len(results)):
        q = make_question(instance['question'], instance['options'])
        prompt = ADD_MISTAKE_FEWSHOT.format(question=q, sentence=instance['cot_step'])
        response = query_api(prompt, client)

        answer = response["text"]
        augmented_results[idx]['mistake_cot_step'] = answer
        augmented_results[idx]['mistake_model'] = response["model"]

    store_jsonl(augmented_results, path_to_store)